In [ ]:
import os
import json
from datetime import datetime

# Create results directory if it doesn't exist
os.makedirs("../results/test_results", exist_ok=True)

# Save results to CSV
csv_path = "../results/test_results/test_results.csv"
results_df.to_csv(csv_path, index=False)
print(f"✓ Test results saved to {csv_path}")

# Save detailed results to JSON
json_path = "../results/test_results/test_results_detailed.json"
json_results = {
    "timestamp": datetime.now().isoformat(),
    "model_path": model_path,
    "total_tests": len(test_cases),
    "total_inference_time": total_time,
    "average_inference_time": total_time / len(test_cases),
    "results": results
}

with open(json_path, "w") as f:
    json.dump(json_results, f, indent=2)
print(f"✓ Detailed results saved to {json_path}")

# Print summary statistics
print("\n" + "="*80)
print("TEST SUMMARY")
print("="*80)
print(f"Total Tests: {len(test_cases)}")
print(f"Total Inference Time: {total_time:.2f}s")
print(f"Average Inference Time: {total_time/len(test_cases):.2f}s")
print(f"Model Path: {model_path}")

## Save Test Results

In [ ]:
def ask_question(context, question):
    """Ask the model a question"""
    prompt = format_qa_prompt(context, question)
    
    start_time = time.time()
    output = text_generator(
        prompt,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
        top_p=0.95,
    )
    inference_time = time.time() - start_time
    
    response = output[0]["generated_text"].split("Answer:")[-1].strip()
    
    print(f"Context: {context}\n")
    print(f"Question: {question}\n")
    print(f"Answer: {response}\n")
    print(f"Inference time: {inference_time:.2f}s")
    
    return response

# Example custom question
context_custom = "Python is a programming language created by Guido van Rossum in 1991."
question_custom = "Who created Python?"

print("Custom Question Test:")
print("="*80)
response = ask_question(context_custom, question_custom)

## Custom Inference

Teste o modelo com seus próprios prompts:

In [ ]:
def format_qa_prompt(context, question):
    """Format QA prompt for inference"""
    return f"""You are a helpful QA assistant. Answer the question based on the context provided.

Context: {context}

Question: {question}

Answer:"""

# Run inference tests
results = []
total_time = 0

for idx, row in test_df.iterrows():
    prompt = format_qa_prompt(row["context"], row["question"])
    
    # Measure inference time
    start_time = time.time()
    output = text_generator(
        prompt,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
        top_p=0.95,
    )
    inference_time = time.time() - start_time
    total_time += inference_time
    
    response = output[0]["generated_text"].split("Answer:")[-1].strip()
    
    result = {
        "test_id": row["id"],
        "question": row["question"],
        "expected_answer": row["expected"],
        "generated_answer": response,
        "inference_time": inference_time,
    }
    results.append(result)
    
    print(f"\n{'='*80}")
    print(f"Test {row['id']}")
    print(f"{'='*80}")
    print(f"Question: {row['question']}")
    print(f"Expected: {row['expected']}")
    print(f"Generated: {response}")
    print(f"Inference time: {inference_time:.2f}s")

print(f"\n{'='*80}")
print(f"Total inference time: {total_time:.2f}s")
print(f"Average time per test: {total_time/len(test_cases):.2f}s")

# Create results dataframe
results_df = pd.DataFrame(results)
print("\nResults Summary:")
print(results_df[["test_id", "question", "inference_time"]])

## Run Inference Tests

In [ ]:
import pandas as pd

# Define test cases
test_cases = [
    {
        "id": 1,
        "context": "The Statue of Liberty is a colossal neoclassical sculpture located on Liberty Island in New York Harbor.",
        "question": "Where is the Statue of Liberty located?",
        "expected": "Liberty Island in New York Harbor"
    },
    {
        "id": 2,
        "context": "Albert Einstein was a theoretical physicist who developed the theory of relativity.",
        "question": "What did Albert Einstein develop?",
        "expected": "the theory of relativity"
    },
    {
        "id": 3,
        "context": "The Amazon Rainforest produces about 20% of the world's oxygen.",
        "question": "What percentage of the world's oxygen does the Amazon produce?",
        "expected": "20%"
    },
    {
        "id": 4,
        "context": "Python is a high-level programming language known for its simplicity and readability.",
        "question": "What is Python known for?",
        "expected": "simplicity and readability"
    },
    {
        "id": 5,
        "context": "The Great Wall of China was built over many centuries for defense purposes.",
        "question": "Why was the Great Wall of China built?",
        "expected": "defense purposes"
    },
]

test_df = pd.DataFrame(test_cases)
print(f"Test cases defined: {len(test_cases)}")
print(test_df)

## Define Test Cases

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch
import time

# Load the fine-tuned model
model_path = "../results/fine-tuned-model"
device_id = 0 if torch.cuda.is_available() else -1

print(f"Loading model from {model_path}...")
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

# Create text generation pipeline
text_generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=device_id,
    max_new_tokens=256,
)

print("✓ Model loaded successfully!")
print(f"Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

## Load Fine-tuned Model

In [ ]:
import subprocess
import sys

# Install testing dependencies
packages = [
    "transformers",
    "torch",
    "peft",
    "numpy",
    "pandas",
]

for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("✓ Testing dependencies installed!")

# Model Testing and Evaluation

Este notebook testa o modelo fine-tuned com diversos prompts de QA para validar a qualidade das respostas.

## Pipeline de Testes
1. Carregar modelo fine-tuned
2. Definir prompts de teste
3. Executar inferências
4. Avaliar respostas
5. Gerar relatório de performance